순서: `ner_documents -> ner_rule_results -> ner_runs -> ner_reviews`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

#--DATA_VERSION: 적재 대상 데이터 버전 수정 필요------------------------------------------- 
DATA_VERSION = '20261110_175720'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / DATA_VERSION
RULE_INPUT_DIR = PROJECT_ROOT / 'data' / 'filter' / 'rule_detect_data' / DATA_VERSION

#--DOCUMENT_ROOT: 적재 대상 데이터 경로 수정 필요------------------------------------------- 
DOCUMENT_ROOT = PROJECT_ROOT / 'data' / 'test' / 'test_data_v2'
DOCUMENT_FILES = sorted(DOCUMENT_ROOT.glob('*.json'))
# DOCUMENT_ROOT = PROJECT_ROOT / 'data' / 'org' / DATA_VERSION
# DOCUMENT_FILES = sorted(DOCUMENT_ROOT.glob('*.json'))

#--RUN_PATHS: 적재 대상 데이터 경로 수정 필요----------------------------------------------- 
RUN_PATHS = sorted(OUTPUT_ROOT.glob('run_20261111_*'))
RUN_PATHS

[WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_181549'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_185228'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_195129'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_195746'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_201907'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_202437')]

#### 0. 테이블 초기화 및 생성

재적재할 때만 `RESET_TABLES = True`로 변경합니다.

In [2]:
from src.config import get_postgre_connect
from src.postgresql.insert import ner_documents, ner_rule_results, ner_runs, ner_reviews, ner_corrected_sentences

#--RESET_TABLES: 초기에 테이블 구조가 변경될 시 True로 변경
RESET_TABLES = True

def reset_tables():
    with get_postgre_connect() as connection:
        connection.execute('DROP TABLE IF EXISTS ner_rule_results, ner_reviews, ner_corrected_sentences, ner_documents, ner_runs CASCADE')
    ner_runs.create_table()
    ner_documents.create_table()
    ner_corrected_sentences.create_table()
    ner_reviews.create_table()
    ner_rule_results.create_table()
    print('테이블 초기화 및 생성 완료')

if RESET_TABLES:
    reset_tables()
else:
    print('초기화 생략')

ner_runs 테이블 생성 완료
ner_documents 테이블 생성 완료
ner_corrected_sentences 테이블 생성 완료
ner_reviews 테이블 생성 완료
ner_rule_results 테이블 생성 완료
테이블 초기화 및 생성 완료


##### 1. ner_documents

테스트 예시 파일은 `data/test/test_data_v2/강원일보.json`이며, 실제 적재는 해당 폴더의 JSON 전체를 처리합니다. 운영에서는 `data/org/원천데이터폴더명/*.json`으로 바꿉니다.

In [3]:
for document_path in DOCUMENT_FILES:
    count = ner_documents.insert_ner_documents(document_path, data_version=DATA_VERSION)
    print(document_path, count)

c:\work\ner\data\test\test_data_v2\강원일보.json 71
c:\work\ner\data\test\test_data_v2\경기일보.json 27
c:\work\ner\data\test\test_data_v2\경향신문.json 106
c:\work\ner\data\test\test_data_v2\노컷뉴스.json 91
c:\work\ner\data\test\test_data_v2\뉴스1.json 35
c:\work\ner\data\test\test_data_v2\뉴스핌.json 41
c:\work\ner\data\test\test_data_v2\뉴시스.json 30
c:\work\ner\data\test\test_data_v2\디지털타임스.json 36
c:\work\ner\data\test\test_data_v2\마이데일리.json 19
c:\work\ner\data\test\test_data_v2\부산일보.json 35
c:\work\ner\data\test\test_data_v2\서울신문.json 34
c:\work\ner\data\test\test_data_v2\세계일보.json 88
c:\work\ner\data\test\test_data_v2\스포츠경향.json 21
c:\work\ner\data\test\test_data_v2\스포츠투데이.json 9
c:\work\ner\data\test\test_data_v2\아시아투데이.json 46
c:\work\ner\data\test\test_data_v2\이투데이.json 10
c:\work\ner\data\test\test_data_v2\일간투데이.json 46
c:\work\ner\data\test\test_data_v2\충청일보.json 54


##### 2. ner_rule_results

룰 PKL은 `data_version` 기준으로 한 번만 적재합니다.

In [7]:
ner_rule_results.insert_ner_rule_results(RULE_INPUT_DIR, data_version=DATA_VERSION)

3002

##### 3. ner_runs

예시: `outputs/.../run_20261111_185228/ner_runs.parquet`

In [8]:
RUN_PATHS

[WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_181549'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_185228'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_195129'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_195746'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_201907'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_202437')]

In [9]:
ner_runs.insert_ner_runs(RUN_PATHS[0] / 'ner_runs.parquet')

1

In [10]:
ner_runs.insert_ner_runs(RUN_PATHS[1] / 'ner_runs.parquet')

1

In [11]:
ner_runs.insert_ner_runs(RUN_PATHS[2] / 'ner_runs.parquet')

1

In [12]:
ner_runs.insert_ner_runs(RUN_PATHS[3] / 'ner_runs.parquet')

1

In [13]:
ner_runs.insert_ner_runs(RUN_PATHS[4] / 'ner_runs.parquet')

1

In [14]:
ner_runs.insert_ner_runs(RUN_PATHS[5] / 'ner_runs.parquet')

1

##### 4. ner_reviews

예시: `run_20261111_185228/reports/강원일보/ner_reviews.parquet`
각 run의 `reports/*/ner_reviews.parquet`를 언론사별로 적재합니다.

In [15]:
def insert_reviews_for_run(run_path):
    review_paths = sorted(run_path.glob('reports/*/ner_reviews.parquet'))
    if not review_paths:
        raise FileNotFoundError(f'리뷰 Parquet가 없습니다: {run_path}')
    total = 0
    for review_path in review_paths:
        total += ner_reviews.insert_ner_reviews(review_path)
        print(review_path, '적재 완료')
    return total

In [16]:
insert_reviews_for_run(RUN_PATHS[0])

c:\work\ner\outputs\20261110_175720\run_20261111_181549\reports\강원일보\ner_reviews.parquet 적재 완료


196

In [17]:
insert_reviews_for_run(RUN_PATHS[1])

c:\work\ner\outputs\20261110_175720\run_20261111_185228\reports\강원일보\ner_reviews.parquet 적재 완료


196

In [19]:
insert_reviews_for_run(RUN_PATHS[2])

c:\work\ner\outputs\20261110_175720\run_20261111_195129\reports\강원일보\ner_reviews.parquet 적재 완료


196

In [20]:
insert_reviews_for_run(RUN_PATHS[3])

c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\경기일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\경향신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\노컷뉴스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\뉴스1\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\뉴스핌\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\뉴시스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\디지털타임스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\마이데일리\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\부산일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\서울신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_1

2121

In [21]:
insert_reviews_for_run(RUN_PATHS[4])

c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\경기일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\경향신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\노컷뉴스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\뉴스1\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\뉴스핌\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\뉴시스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\디지털타임스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\마이데일리\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\부산일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\서울신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_2

2120

In [22]:
insert_reviews_for_run(RUN_PATHS[5])

c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\경기일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\경향신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\노컷뉴스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\뉴스1\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\뉴스핌\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\뉴시스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\디지털타임스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\마이데일리\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\부산일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\서울신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_2

2120